# Map Reduce

Eerst maken we een aparte bucket aan in de minIO cluster.
Deze bucket gebruiken we voor alle opslag in deze notebook zodat er geen conflicten zijn met reeds bestaande files en/of er data overschreven kan worden.
Dit kunnen we doen met onderstaande code.

In [1]:
from minio import Minio
import os

client = Minio(
    endpoint="minio1:9000",
    access_key="bigdata",
    secret_key="bigdata123",
    secure=False,
    region="eu-west-1"
)

# Check verbinding
try:
    print(client.list_buckets())
    print("Verbonden met MinIO")
except Exception as e:
    print("Fout bij verbinden:", e)

bucket_name = "01-mapreduce"
keep_files = {"input.txt", "titanic.csv"}

# Stap 1: bucket aanmaken als die niet bestaat
if not client.bucket_exists(bucket_name):
    client.make_bucket(bucket_name)
    print(f"Bucket '{bucket_name}' aangemaakt")

    # Stap 2: upload files
    for fname in keep_files:
        # Upload bestand uit lokale map als het ontbreekt
        if os.path.exists(fname):
            client.fput_object(bucket_name, fname, fname)
            print(f"Geüpload: {fname}")
        else:
            print(f"Let op: {fname} bestaat niet lokaal en kon niet geüpload worden.")
else:
    print(f"Bucket '{bucket_name}' bestaat al")

    # Stap 3: bucket leegmaken behalve keep1.txt en keep2.txt
    for obj in client.list_objects(bucket_name):
        if obj.object_name not in keep_files:
            client.remove_object(bucket_name, obj.object_name)
            print(f"Verwijderd: {obj.object_name}")

[Bucket(name='01-mapreduce', creation_date=datetime.datetime(2026, 3, 3, 13, 11, 43, 563000, tzinfo=datetime.timezone.utc)), Bucket(name='02-spark', creation_date=datetime.datetime(2026, 3, 10, 13, 11, 25, 375000, tzinfo=datetime.timezone.utc)), Bucket(name='mijnclibucket', creation_date=datetime.datetime(2026, 3, 6, 13, 6, 30, 467000, tzinfo=datetime.timezone.utc)), Bucket(name='mijnpythonbucket', creation_date=datetime.datetime(2026, 2, 24, 14, 8, 57, 985000, tzinfo=datetime.timezone.utc))]
Verbonden met MinIO
Bucket '01-mapreduce' bestaat al
Verwijderd: output.txt
Verwijderd: output_oef1.txt
Verwijderd: output_oef3.txt
Verwijderd: output_oef4.txt
Verwijderd: output_titanic.txt


## Wat is MapReduce

MapReduce is een programmeermodel om eenvoudig distributed data te verwerken.
Het is belangrijk om te realiseren dat de programma's die je hier schrijft een parallel uitgevoerd worden op verschillende stukjes data (De map-fase) om daarna in de reduce-fase tot een finale output teruggebracht te worden.
Er gebeuren 5 stappen bij het uitvoeren van een MapReduce programma
* Bepalen op welke nodes de code uitgevoerd wordt (wordt door YARN gedaan afhankelijk van de locatie van de blocks)
* Uitvoeren van de Map-code (Geschreven door de developer naar eigen wens)
* Shuffle, ouput van de map-fase doorsturen naar andere nodes die de resultaten gaan reduceren (Automatisch)
* uitvoeren van de Reduce-code (Geschreven door de developer naar eigen wens)
* Combineren van de reduce output tot 1 gehele/finale output (Automatisch)

Uit bovenstaand stappenplan is het duidelijk dat er twee zaken moeten geimplementeerd worden bij het schrijven van een MapReduce toepassing. 
De andere stappen zijn identiek voor alle mapreduce applicaties.
Het gaat hierbij over 

* de mapper functie die bepaalt welke (key,value) paren uit een lijn tekst gehaald worden
* de reduce functie die bepaalt hoe alle values voor een key gecombineerd worden tot een eindresultaat.

Er bestaan een reeks packages die de andere functionaliteiten voor jou inbouwen. In deze notebook en voor dit vak gaan we echter werken met een zelfgemaakte implementatie van het map-reduce algoritme.

Het basis-stappenplan van de mapreduce applicatie waarbij een file uitgelezen wordt van de MinIO cluster, deze file verwerkt wordt en de output teruggeschreven wordt naar de cluster is als volgt.

In [4]:
import s3fs
from collections import defaultdict

# maak een connectie met het filesysteem
fs = s3fs.S3FileSystem(
    key='bigdata',
    secret='bigdata123',
    client_kwargs={'endpoint_url': 'http://minio1:9000'}
)

input_file = 's3://01-mapreduce/input.txt' #s3 -> protocol / bucket_name / filename
output_file = 's3://01-mapreduce/output.txt'

# lees de input
with fs.open(input_file, 'r') as f:
    lines = f.readlines()

# mapper
def mapper(line):
    # werkt lijn per lijn
    return [(word, 1) for word in line.strip().split(' ')]

mapped = []
for line in lines:
    mapped.extend(mapper(line))
    # extend is niet gelijk aan append
    
# shuffle
shuffle_dict = defaultdict(list)
for key, value in mapped:
    # volgende lijnen niet nodig door defaultdict
    #if key not in shuffle_dict.keys():
        #shuffle_dict[key] = []
    shuffle_dict[key].append(value)
    
# reducer
reduced = {}
for key, values in shuffle_dict.items():
    reduced[key] = sum(values)

# wegschrijven output
with fs.open(output_file, 'w') as f:
    for word, count in reduced.items():
        f.write(f"{word}\t{count}\n")

Voor je verder gaat, controleer of het bestand in de correcte bucket geplaatst is. Download ook het bestand en controleer de output.

## Oefeningen

Voor we naar de oefeningen gaan, gaan we bovenstaande code verbeteren.
Maak een MapReduceJob klasse die de volgende functies heeft

* Een constructor met twee parameters (mapper en reducer) welke de uit te voeren functies zijn
* Een run stap met een parameter lines (de lijnen tekst in een bestand)
  * Deze functie voert het mapreduce proces uit
  * Stuur lines door de mapper functie
  * Shuffle het resultaat zodat we van
    * [(key1, value1), (key2, value2), ...] naar [(key1, [value1, value3, ...]), (key2, [value2, value4, ...])]
  * Stuur dit resultaat door de reducer functie
  * Return het resultaat.

Maak met behulp van de voorgaande klasse, de nodige mapreduce applicaties voor de volgende zaken te berekenen (je moet de output niet bewaren in een file):
* De gemiddelde woordlengte
* Het aantal keer dat elk karakter voorkomt
* Het aantal woorden dat begint met elke letter
* Het aantal woorden in de tekst

## Werken met MapReduce voor gestructureerde data

Mapreduce kan ook gebruikt worden om te werken met gestructureerde data (bijvoorbeeld een csv) waar elke rij 1 data-element voorsteld.
Het is hier echter wel belangrijk dat alle data op 1 lijn gecombineerd wordt dus multiline csv's, jsons of xml bestanden kunnen niet verwerkt worden.

Onderstaande code is een voorbeeld voor hoe je een csv kan uitlezen en een aantal statistieken kan berekenen. Hierin leer je vooral:
* Hoe de csv lijn per lijn te verwerken en kolommen te detecteren
* Hoe de header rij eruit filteren
* Hoe meerdere berekeningen op een iterator te doen

We gebruiken hiervoor de titanic.csv file. Let op dat die naam van de passagier hierbij komma's kan bevatten dus dit vereist wat extra aandacht.
Daarnaast gebruiken we de lokale versie van het bestand en niet de geuploadde versie.

We willen de volgende zaken berekenen door middel van 1 map-reduce applicatie:
* Gemiddelde leeftijd
* Percentage overleeft
* Percentage mannelijke passagiers
* Percentage vrouwelijke passagiers die het overleefd hebben: 30% betekend dat 30% van de vrouwelijke passagiers het overleefd hebben

## Verdere oefeningen

Gebruik nu de titanic csv om met behulp van een mapreduce applicatie de volgende zaken te berekenen:
* Het aantal unieke plaatsen waar personen aan boord zijn gekomen (embarked kolom)
* Het aantal ontbrekende waarden in de Cabin kolom
* De volgende statistische waarden voor de ticketprijs (Fare) kolom: min, max, mean, std
* De langste naam van een passagier
* Het aantal passagiers per klasse
* Het totaal aantal passagiers op de titanic